In [1]:
# Load env variables
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
import os

api_key = os.getenv("ANTHROPIC_API_KEY")

if api_key:
    print(f"✅ API key загружен: {api_key[:8]}...{api_key[-4:]}")
else:
    print("❌ API key не найден")

✅ API key загружен: sk-ant-a...sQAA


In [11]:
# Create an Anthropics API client
from anthropic import Anthropic

client = Anthropic()

In [37]:
# Defining helper functions

# Add user message
def add_user_message(messages, text):
    user_message = {
        "role": "user",
        "content": text
    }
    messages.append(user_message)

# Add assistant message
def add_assistant_message(messages, text):
    assistant_message = {
        "role": "assistant",
        "content": text
    }
    messages.append(assistant_message)

def chat(messages, system=None, stop_sequences=None):

    params = {
        "model": "claude-haiku-4-5-20251001",
        "max_tokens": 2000,
        "messages": messages,
    }
    
    if system:
        params["system"] = system
    
    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    response = client.messages.create( **params)
    return response.content[0].text

In [46]:
import json

# Generate test data set

def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
        "format": "json" or "python" or "regex",
        "solution_criteria": "Key criteria for evaluating the solution"
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 6 objects.
    """

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    answer = chat(messages, stop_sequences=["```"])
    return json.loads(answer)

In [ ]:
# Regenerate the test cases dataset
dataset = generate_dataset()

dataset

with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)
    

In [34]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    
    prompt = f"""
    Please solver the following task:
    {test_case["task"]}

    * Respond only with Python, JSON, or plain Regex
    * Do not add any comments or commentary or explanation
    """

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```code")
    answer = chat(messages, stop_sequences=["```"])
    return answer

In [45]:
# Function to grade a test case + output using a model
def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Criteria you should use to evaluate the solution:
<criteria>
{test_case["solution_criteria"]}
</criteria>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
    """

    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

In [24]:
# Functions to validate the output structure
import re
import ast


def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0


def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0


def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0


def grade_syntax(response, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)

In [39]:
def run_test_case(test_case):
    """Calls run_promp, then grades the result"""
    output = run_prompt(test_case)

    # Grading
    model_grade = grade_by_model(test_case, output)
    reasoning = model_grade["reasoning"]
    model_score = model_grade["score"]

    systax_score = grade_syntax(output, test_case)

    score = (model_score + systax_score)/2

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning,
    }

In [40]:
from statistics import mean

def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")

    return results

In [48]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

    results = run_eval(dataset)

    print(json.dumps(results, indent=2))

Average score: 6.416666666666667
[
  {
    "output": "\nimport re\nimport json\nfrom datetime import datetime\n\ndef parse_cloudwatch_log(log_entry):\n    \"\"\"\n    Parse an AWS CloudWatch log entry and extract timestamp, log level, and message\n    \n    Expected format: YYYY-MM-DD HH:MM:SS.fff [LEVEL] message\n    \"\"\"\n    pattern = r'^(\\d{4}-\\d{2}-\\d{2}\\s\\d{2}:\\d{2}:\\d{2}\\.\\d{3})\\s\\[([A-Z]+)\\]\\s(.+)$'\n    \n    match = re.match(pattern, log_entry.strip())\n    \n    if match:\n        timestamp, level, message = match.groups()\n        return {\n            'timestamp': timestamp,\n            'log_level': level,\n            'message': message\n        }\n    \n    return None\n\n\nif __name__ == \"__main__\":\n    test_logs = [\n        \"2024-01-15 10:30:45.123 [INFO] Application started successfully\",\n        \"2024-01-15 10:30:46.456 [ERROR] Database connection failed\",\n        \"2024-01-15 10:30:47.789 [WARNING] High memory usage detected\"\n    ]\n    \

In [42]:
print(json.dumps(results, indent=2))

[
  {
    "output": "\nimport re\n\ndef parse_s3_bucket_name(s3_uri):\n    match = re.match(r's3://([^/]+)', s3_uri)\n    if match:\n        return match.group(1)\n    return None\n",
    "test_case": {
      "task": "Parse an AWS S3 bucket name from a full S3 object URI (s3://bucket-name/path/to/object)",
      "format": "regex"
    },
    "score": 8.5,
    "reasoning": "The solution correctly solves the basic requirement with a straightforward regex approach. However, it lacks robustness for production use. Adding input validation, handling common S3 URI variants (s3a://, query params), and documenting assumptions would improve reliability. The code is functional but incomplete for real-world AWS scenarios."
  },
  {
    "output": "\n{\n  \"FunctionName\": \"my-lambda-function\",\n  \"Runtime\": \"python3.11\",\n  \"Role\": \"arn:aws:iam::123456789012:role/lambda-execution-role\",\n  \"Handler\": \"index.handler\",\n  \"Timeout\": 300,\n  \"MemorySize\": 512,\n  \"Environment\": {\n 